# load dependecies

In [ ]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import copy
import yasa
from mne.filter import resample
import pynapple as nap
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import normalize
import requests
from io import BytesIO
import sails
from ssqueezepy import cwt
import re
from scipy.stats import entropy
import os, sys

import scipy
from scipy import signal
from scipy.interpolate import griddata
from scipy.signal import correlate
from scipy.stats import pearsonr
from scipy.fft import fft
from scipy.spatial.distance import euclidean
from scipy.signal import spectrogram
from scipy.io import loadmat
import scipy.fft
import scipy.stats
import scipy.io as sio
from scipy.signal import hilbert

import emd as emd
import emd.sift as sift
import emd.spectra as spectra

from neurodsp.sim import sim_combined
from neurodsp.plts import plot_time_series, plot_timefrequency
from neurodsp.utils import create_times
from neurodsp.timefrequency.wavelets import compute_wavelet_transform
from neurodsp.filt import filter_signal

# Load required libraries
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from neurodsp.filt import filter_signal, filter_signal_fir, design_fir_filter
import emd
import pandas as pd
from sklearn.preprocessing import Normalizer
from tqdm import tqdm
import plotly.express as px
import copy
import umap.umap_ as umap
from scipy.spatial import cKDTree
import pickle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

## UTILS

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

from scipy.io import loadmat
import numpy as np
from neurodsp.filt import filter_signal
import copy
import emd
from scipy.spatial import cKDTree
from tqdm import tqdm

sns.set(style='white', context='notebook')

In [ ]:
path_to_config = '/Users/amir/Desktop/for Abdel/emd_masksift_CA1_config.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

# Preprocessing

In [ ]:
def extract_lfp_by_pt_intervals(lfp, fs, interval):
    rem_lfp = []
    for ii in range(len(interval)):
        start_idx = int(interval.loc[ii, 'start'] * fs)
        end_idx = int(interval.loc[ii, 'end'] * fs)
        rem_lfp.append(np.array(lfp[start_idx:end_idx]))
    return rem_lfp


In [ ]:
# preprocessing
def extract_cycle_info(imfs, imf_frequencies):

  waveforms = pd.DataFrame()
  all_trials = pd.DataFrame()
  raw_wavelets = []
  all_FPPs = []

  theta_range = [5, 12]
  frequencies = np.arange(15, 141, 1)  # Logarithmically spaced frequencies from 15 to 300 Hz
  angles=np.linspace(-180,180,19)
  fs = 1000

  for idx, imf in enumerate(imfs):
    cycle_data = get_cycle_data(imf[:, 5], fs)

    amp_thresh = np.percentile(cycle_data['IA'], 25) # higher than 25th percentile of the data
    lo_freq_duration = fs/5  # restrict the analysis to 5-12 Hz
    hi_freq_duration = fs/12

    conditions = ['is_good==1',
                        f'duration_samples<{lo_freq_duration}',
                        f'duration_samples>{hi_freq_duration}',
                        f'max_amp>{amp_thresh}']
    print(len(cycle_data['theta_imf']))
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    
    # Check if any cycles satisfy the conditions
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        print("No cycles satisfy the conditions.")
        return pd.DataFrame(), pd.DataFrame(), []
    
    subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_cycles_df['index'].values

    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycles_inds = arrange_cycle_inds(all_cycles_inds)

    freqs = imf_frequencies[idx]
    sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
    supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)

    # # Corrected Wavelet Transform Computation
    raw_data=sails.wavelet.morlet(supra_theta_sig, freqs=frequencies, sample_rate=fs, ncycles=5,ret_mode='power', normalise='simple')
    raw_wavelets.append(raw_data)
    supraPlot = scipy.stats.zscore(raw_data, axis=1)
    FPP = bin_tf_to_fpp(cycles_inds, supraPlot, bin_count=19)
    all_FPPs.append(FPP)

    # Compute mode frequency for each cycle
    mode_freqs, entropies = compute_mode_frequency_and_entropy(FPP, frequencies, angles)

    all_waveforms, _ = emd.cycles.phase_align(cycle_data['IP'], cycle_data['theta_imf'],
                                                            cycles=all_cycles.iterate(through='subset'), npoints=100)
    all_waveforms = pd.DataFrame(all_waveforms.T)

    waveforms = pd.concat([waveforms, all_waveforms])

    trial = all_cycles.get_metric_dataframe(subset=True)
    trial['mode_freqs'] = mode_freqs
    trial['entropy'] = entropies
    all_trials = pd.concat([all_trials, trial])

  return waveforms, all_trials, all_FPPs

In [ ]:
path_to_hpc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/HPC_100_CH4_0.continuous.mat'
path_to_pfc = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/PFC_100_CH47_0.continuous.mat'
path_to_hypno = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition/HomeCageHC/OS_Ephys_RGS14_Rat1_57986_SD4_HC_01_08_2018/2018-08-01_15-49-15_Post-Trial5/2018-08-01_15-49-15_Post-Trial5_concatenated-states.mat'

In [ ]:
fs = 1000
theta_range = [5, 12]

lfpHPC, hypno, _ = get_data(path_to_hpc, path_to_hypno)
lfpPFC, _, _ = get_data(path_to_pfc, path_to_hypno, type='pfc')
phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=1000)

In [ ]:
tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, tonic_interval, config, return_imfs_freqs=True)
phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, phasic_interval, config, return_imfs_freqs=True)


tonic_theta_imfs  = [imf[:, 5] for imf in tonic_imfs]
phasic_theta_imfs = [imf[:, 5] for imf in phasic_imfs]

In [ ]:
phasic_waveforms, phasic_trials, phasic_fpps = extract_cycle_info(phasic_imfs, phasic_freqs)
tonic_waveforms, tonic_trials, tonic_fpps = extract_cycle_info(tonic_imfs, tonic_freqs)

In [ ]:
pfc_phasic_rem_lfp = extract_lfp_by_pt_intervals(lfpPFC, fs, phasic_interval)
pfc_tonic_rem_lfp = extract_lfp_by_pt_intervals(lfpPFC, fs, tonic_interval)

# Field-Field Pairwise Phase Consistency (PPC)
# Following Zhang et al. (2019) - eLife

In [ ]:
def compute_field_field_ppc(hpc_imfs, pfc_lfp_segments, fs=1000,
                            frequencies=np.arange(15, 141, 1),
                            n_phase_bins=20, n_cycles_wavelet=5):
    """
    Compute field-field PPC between HPC and PFC following Zhang et al. (2019).

    For each theta cycle (detected from HPC theta IMF):
      1. Compute complex Morlet wavelet transform of both HPC and PFC raw LFP
      2. Compute cross-spectrum: WC = W_hpc * conj(W_pfc)
      3. Bin cross-spectrum into theta phase bins within the cycle
      4. Extract phase difference: W_k(f, phi) = angle(B_k(f, phi))

    PPC is then computed across all theta cycles:
      PPC(f, phi) = (|sum_k exp(i * W_k)|^2 - N) / (N * (N-1))

    Parameters
    ----------
    hpc_imfs : list of ndarray
        HPC IMFs per interval, each shape (n_samples, n_imfs).
        Broadband HPC signal is reconstructed by summing all IMFs.
        Theta cycles are detected from IMF index 5.
    pfc_lfp_segments : list of ndarray
        PFC raw LFP per interval, each shape (n_samples,).
    fs : int
        Sampling frequency in Hz.
    frequencies : ndarray
        Frequencies for wavelet transform.
    n_phase_bins : int
        Number of theta phase bins per cycle (Zhang used 20).
    n_cycles_wavelet : int
        Number of cycles for Morlet wavelet.

    Returns
    -------
    ppc : ndarray, shape (n_freqs, n_phase_bins)
        Pairwise phase consistency matrix. Values range from -1 to 1.
    frequencies : ndarray
        Frequency vector used.
    n_theta_cycles : int
        Total number of theta cycles used in the computation.
    """
    all_W = []  # phase-difference matrices, one per theta cycle

    for idx in tqdm(range(len(hpc_imfs)), desc='Computing PPC across intervals'):
        imf = hpc_imfs[idx]
        pfc = pfc_lfp_segments[idx]

        # Reconstruct broadband HPC signal from IMFs (EMD is a complete decomposition)
        hpc = np.sum(imf, axis=1)

        # Ensure both signals have the same length
        min_len = min(len(hpc), len(pfc))
        hpc, pfc = hpc[:min_len], pfc[:min_len]
        theta_imf = imf[:min_len, 5]

        # --- Detect theta cycles from HPC theta IMF ---
        cycle_data = get_cycle_data(theta_imf, fs)
        amp_thresh = np.percentile(cycle_data['IA'], 25)
        conditions = [
            'is_good==1',
            f'duration_samples<{fs / 5}',
            f'duration_samples>{fs / 12}',
            f'max_amp>{amp_thresh}'
        ]
        all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
        if all_cycles is None or all_cycles.chain_vect.size == 0:
            continue

        subset_df = all_cycles.get_metric_dataframe(subset=True)
        cycle_inds = arrange_cycle_inds(
            get_cycle_inds(all_cycles, subset_df['index'].values)
        )  # shape (n_cycles, 2) with [start_sample, end_sample]

        # --- Complex Morlet wavelet transforms --- shape: (n_freqs, n_times)
        hpc_cwt = sails.wavelet.morlet(
            hpc, freqs=frequencies, sample_rate=fs,
            ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
        )
        pfc_cwt = sails.wavelet.morlet(
            pfc, freqs=frequencies, sample_rate=fs,
            ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
        )

        # Cross-spectrum: (n_freqs, n_times)
        cross = hpc_cwt * np.conj(pfc_cwt)

        # --- For each theta cycle, bin cross-spectrum into phase bins ---
        for c in range(len(cycle_inds)):
            start, end = cycle_inds[c]
            cycle_len = end - start

            if end > min_len or cycle_len < n_phase_bins:
                continue

            cs_cycle = cross[:, start:end]  # (n_freqs, cycle_len)

            # Divide cycle into n_phase_bins equal time bins
            # (approximates equal theta-phase bins within one cycle)
            bin_edges = np.linspace(0, cycle_len, n_phase_bins + 1, dtype=int)
            B_k = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
            for pb in range(n_phase_bins):
                b_s, b_e = bin_edges[pb], bin_edges[pb + 1]
                if b_e > b_s:
                    B_k[:, pb] = np.mean(cs_cycle[:, b_s:b_e], axis=1)

            # Phase difference between HPC and PFC at each (freq, phase_bin)
            W_k = np.angle(B_k)
            all_W.append(W_k)

    N = len(all_W)
    print(f'Total theta cycles used for PPC: {N}')

    if N < 2:
        print('Not enough theta cycles to compute PPC.')
        return None, frequencies, N

    # --- Compute PPC using efficient vectorized formula ---
    # PPC(f, phi) = (|sum_k e^{i*W_k}|^2 - N) / (N * (N - 1))
    sum_vec = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
    for W_k in all_W:
        sum_vec += np.exp(1j * W_k)

    ppc = (np.abs(sum_vec)**2 - N) / (N * (N - 1))

    return ppc, frequencies, N

In [ ]:
# Also extract raw HPC LFP segments for cross-spectrum computation
hpc_phasic_rem_lfp = extract_lfp_by_pt_intervals(lfpHPC, fs, phasic_interval)
hpc_tonic_rem_lfp = extract_lfp_by_pt_intervals(lfpHPC, fs, tonic_interval)

In [ ]:
# Compute PPC for phasic REM
ppc_phasic, ppc_freqs, n_cycles_phasic = compute_field_field_ppc(
    phasic_imfs, pfc_phasic_rem_lfp, fs=fs
)
print(f'Phasic REM: {n_cycles_phasic} theta cycles')

In [ ]:
# Compute PPC for tonic REM
ppc_tonic, _, n_cycles_tonic = compute_field_field_ppc(
    tonic_imfs, pfc_tonic_rem_lfp, fs=fs
)
print(f'Tonic REM: {n_cycles_tonic} theta cycles')

In [ ]:
# --- Plot PPC(f, phi) heatmaps for phasic and tonic REM ---
phase_bins = np.linspace(-180, 180, 21)  # 20 bins
phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Phasic REM PPC
if ppc_phasic is not None:
    im0 = axes[0].pcolormesh(phase_centers, ppc_freqs, ppc_phasic,
                              cmap='RdBu_r', shading='auto', vmin=-0.05, vmax=0.15)
    axes[0].set_xlabel('Theta phase (deg)')
    axes[0].set_ylabel('Frequency (Hz)')
    axes[0].set_title(f'Phasic REM PPC (n={n_cycles_phasic})')
    plt.colorbar(im0, ax=axes[0], label='PPC')

# Tonic REM PPC
if ppc_tonic is not None:
    im1 = axes[1].pcolormesh(phase_centers, ppc_freqs, ppc_tonic,
                              cmap='RdBu_r', shading='auto', vmin=-0.05, vmax=0.15)
    axes[1].set_xlabel('Theta phase (deg)')
    axes[1].set_ylabel('Frequency (Hz)')
    axes[1].set_title(f'Tonic REM PPC (n={n_cycles_tonic})')
    plt.colorbar(im1, ax=axes[1], label='PPC')

# Difference: Phasic - Tonic
if ppc_phasic is not None and ppc_tonic is not None:
    ppc_diff = ppc_phasic - ppc_tonic
    vmax_diff = np.max(np.abs(ppc_diff)) * 0.8
    im2 = axes[2].pcolormesh(phase_centers, ppc_freqs, ppc_diff,
                              cmap='RdBu_r', shading='auto',
                              vmin=-vmax_diff, vmax=vmax_diff)
    axes[2].set_xlabel('Theta phase (deg)')
    axes[2].set_ylabel('Frequency (Hz)')
    axes[2].set_title('Phasic - Tonic')
    plt.colorbar(im2, ax=axes[2], label='ΔPPC')

plt.tight_layout()
plt.show()

In [ ]:
# --- PPC averaged across all theta phase bins -> PPC(f) ---
# This gives a frequency-resolved summary of HPC-PFC phase consistency

fig, ax = plt.subplots(figsize=(8, 4))

if ppc_phasic is not None:
    ppc_phasic_avg = np.mean(ppc_phasic, axis=1)
    ax.plot(ppc_freqs, ppc_phasic_avg, label=f'Phasic (n={n_cycles_phasic})', color='red', linewidth=1.5)

if ppc_tonic is not None:
    ppc_tonic_avg = np.mean(ppc_tonic, axis=1)
    ax.plot(ppc_freqs, ppc_tonic_avg, label=f'Tonic (n={n_cycles_tonic})', color='blue', linewidth=1.5)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC (phase-averaged)')
ax.set_title('HPC-PFC Field-Field PPC: Phasic vs Tonic REM')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## look into trials

In [ ]:
def find_file_local(directory, prefix):
    """Auto-discover a .mat file by prefix (e.g. 'HPC_100', 'PFC_100', 'states')."""
    for fname in os.listdir(directory):
        low = fname.lower()
        if low.startswith(prefix.lower()) and low.endswith('.mat'):
            return os.path.join(directory, fname)
        if prefix == 'states' and 'states' in low and low.endswith('.mat'):
            return os.path.join(directory, fname)
    return None


def compute_per_trial_ppc(rat_ids, base_path='/Users/amir/Desktop/for Abdel/RGS/DatabyCondition',
                          fs=1000, frequencies=np.arange(15, 141, 1),
                          n_phase_bins=20, n_cycles_wavelet=5,
                          min_cycles=2, save_path=None):
    """
    Compute PPC(f, phi) per individual post-trial session using compute_field_field_ppc.
    Only keeps trials that have BOTH valid phasic AND tonic PPC (>= min_cycles each).

    Parameters
    ----------
    rat_ids : list of int
        Which rats to process.
    min_cycles : int
        Minimum cycles required for BOTH phasic and tonic to keep a trial.
    Returns
    -------
    trial_results : list of dict
        Each dict has:
          rat_id, condition, sd, trial_number, folder,
          phasic_ppc, phasic_n, tonic_ppc, tonic_n, frequencies, n_phase_bins
        Only trials with both phasic and tonic PPC are included.
    """
    cfg = globals().get("config", None)
    if cfg is None:
        raise RuntimeError("Define `config` (emd SiftConfig) before calling.")

    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    trial_results = []
    skipped_single = 0  # counter for trials skipped because only one state

    for rat_id in rat_ids:
        print(f"\n{'='*60}")
        print(f"  Rat {rat_id} — per-trial PPC")
        print(f"{'='*60}")

        for condition_folder in conditions:
            condition_path = os.path.join(base_path, condition_folder)
            rat_folders = []
            for f in os.listdir(condition_path):
                if not os.path.isdir(os.path.join(condition_path, f)):
                    continue
                m = folder_re.match(f)
                if m and int(m.group(1)) == rat_id:
                    rat_folders.append((f, m))

            for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
                rat_path = os.path.join(condition_path, rat_folder)
                sd_number = m.group(2)
                condition = m.group(3)

                pt_folders = sorted([
                    f for f in os.listdir(rat_path)
                    if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
                ])

                for pt_folder in pt_folders:
                    trial_path = os.path.join(rat_path, pt_folder)
                    hpc_file = find_file_local(trial_path, 'HPC_100')
                    pfc_file = find_file_local(trial_path, 'PFC_100')
                    state_file = find_file_local(trial_path, 'states')

                    if not hpc_file or not pfc_file or not state_file:
                        missing = [x for x, v in [('HPC', hpc_file), ('PFC', pfc_file), ('states', state_file)] if not v]
                        print(f"  [SKIP] {condition_folder}/{pt_folder} — missing: {', '.join(missing)}")
                        continue

                    trial_match = re.search(r'Post[-_]Trial(\d+)', pt_folder)
                    trial_number = int(trial_match.group(1)) if trial_match else -1

                    try:
                        lfpHPC_r, hypno_r, _ = get_data(hpc_file, state_file)
                        lfpPFC_r, _, _ = get_data(pfc_file, state_file, type='pfc')

                        try:
                            phasic_int, tonic_int, lfp_raw_r = extract_pt_intervals(
                                lfpHPC_r, hypno_r, fs=fs)
                        except ValueError:
                            print(f"  [SKIP] {condition_folder}/{pt_folder} — no REM")
                            continue

                        has_phasic = phasic_int is not None and len(phasic_int) > 0
                        has_tonic = tonic_int is not None and len(tonic_int) > 0

                        # Skip if missing either state entirely
                        if not has_phasic or not has_tonic:
                            skipped_single += 1
                            print(f"  [SKIP] {condition_folder}/{pt_folder} — "
                                  f"only {'tonic' if has_tonic else 'phasic'} (need both)")
                            continue

                        trial_entry = {
                            'rat_id': rat_id,
                            'condition': condition_folder,
                            'sd': sd_number,
                            'trial_number': trial_number,
                            'folder': pt_folder,
                            'frequencies': frequencies,
                            'n_phase_bins': n_phase_bins,
                            'phasic_ppc': None, 'phasic_n': 0,
                            'tonic_ppc': None, 'tonic_n': 0,
                        }

                        # --- Phasic ---
                        try:
                            ph_imfs, _, _ = extract_imfs_by_pt_intervals(
                                lfp_raw_r, fs, phasic_int, cfg, return_imfs_freqs=True)
                            pfc_ph = extract_lfp_by_pt_intervals(lfpPFC_r, fs, phasic_int)
                            ppc, freqs, n = compute_field_field_ppc(
                                ph_imfs, pfc_ph, fs=fs,
                                frequencies=frequencies,
                                n_phase_bins=n_phase_bins,
                                n_cycles_wavelet=n_cycles_wavelet)
                            trial_entry['phasic_ppc'] = ppc
                            trial_entry['phasic_n'] = n
                        except Exception as e:
                            print(f"  [WARN] Phasic {condition_folder}/{pt_folder}: {e}")

                        # --- Tonic ---
                        try:
                            to_imfs, _, _ = extract_imfs_by_pt_intervals(
                                lfp_raw_r, fs, tonic_int, cfg, return_imfs_freqs=True)
                            pfc_to = extract_lfp_by_pt_intervals(lfpPFC_r, fs, tonic_int)
                            ppc, freqs, n = compute_field_field_ppc(
                                to_imfs, pfc_to, fs=fs,
                                frequencies=frequencies,
                                n_phase_bins=n_phase_bins,
                                n_cycles_wavelet=n_cycles_wavelet)
                            trial_entry['tonic_ppc'] = ppc
                            trial_entry['tonic_n'] = n
                        except Exception as e:
                            print(f"  [WARN] Tonic {condition_folder}/{pt_folder}: {e}")

                        # Only keep if BOTH phasic and tonic PPC are valid
                        both_valid = (trial_entry['phasic_ppc'] is not None
                                      and trial_entry['tonic_ppc'] is not None
                                      and trial_entry['phasic_n'] >= min_cycles
                                      and trial_entry['tonic_n'] >= min_cycles)

                        if both_valid:
                            trial_results.append(trial_entry)
                            print(f"  [OK]   {condition_folder}/{pt_folder} | "
                                  f"phasic: {trial_entry['phasic_n']} cycles, "
                                  f"tonic: {trial_entry['tonic_n']} cycles")
                        else:
                            skipped_single += 1
                            print(f"  [SKIP] {condition_folder}/{pt_folder} — "
                                  f"phasic: {trial_entry['phasic_n']}, tonic: {trial_entry['tonic_n']} "
                                  f"(need both >= {min_cycles})")

                    except Exception as e:
                        print(f"  [ERROR] {condition_folder}/{pt_folder}: {e}")

    print(f"\nTotal trials kept (both phasic & tonic): {len(trial_results)}")
    print(f"Trials skipped (missing one state): {skipped_single}")

    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(trial_results, f)
        print(f"Saved to {save_path}")

    return trial_results

### per trial - multi rat

In [ ]:
# ---- Run per-trial PPC ----
selected_rats_trials = [1, 3, 6, 9]  # <-- CHANGE THIS

trial_results = compute_per_trial_ppc(
    rat_ids=selected_rats_trials,
    save_path='ppc_per_trial_results.pkl'
)

In [ ]:
# Quick summary table of all trials
summary_rows = []
for t in trial_results:
    summary_rows.append({
        'Rat': t['rat_id'],
        'Condition': t['condition'],
        'SD': t['sd'],
        'Trial': t['trial_number'],
        'Folder': t['folder'],
        'Phasic cycles': t['phasic_n'],
        'Tonic cycles': t['tonic_n'],
        'Has phasic PPC': t['phasic_ppc'] is not None,
        'Has tonic PPC': t['tonic_ppc'] is not None,
    })
trial_summary = pd.DataFrame(summary_rows)
trial_summary

In [ ]:
def plot_trial_ppc_heatmaps(trial_results, rat_id, condition=None, vmin=-0.05, vmax=0.15,
                           save_dir='ppc_trial_figures'):
    """
    Plot PPC(f, phi) heatmaps for individual trials of a specific rat.
    Each trial saved as a separate PNG. All trials guaranteed to have both phasic & tonic.
    """
    os.makedirs(save_dir, exist_ok=True)

    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

    trials = [t for t in trial_results if t['rat_id'] == rat_id]
    if condition is not None:
        trials = [t for t in trials if t['condition'] == condition]

    if not trials:
        print(f"No valid trials for Rat {rat_id}" + (f" condition={condition}" if condition else ""))
        return

    saved_files = []
    for t in trials:
        freqs = t['frequencies']
        label = f"{t['condition']}_SD{t['sd']}_Trial{t['trial_number']}"

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Rat {rat_id} — {t["condition"]}/SD{t["sd"]}/Trial{t["trial_number"]}',
                     fontsize=14, fontweight='bold')

        # Phasic
        im = axes[0].pcolormesh(phase_centers, freqs, t['phasic_ppc'],
                                cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=axes[0], label='PPC')
        axes[0].set_title(f'Phasic (n={t["phasic_n"]})')
        axes[0].set_ylabel('Frequency (Hz)')
        axes[0].set_xlabel('Theta phase (deg)')

        # Tonic
        im = axes[1].pcolormesh(phase_centers, freqs, t['tonic_ppc'],
                                cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=axes[1], label='PPC')
        axes[1].set_title(f'Tonic (n={t["tonic_n"]})')
        axes[1].set_ylabel('Frequency (Hz)')
        axes[1].set_xlabel('Theta phase (deg)')

        # Difference
        diff = t['phasic_ppc'] - t['tonic_ppc']
        vd = max(np.max(np.abs(diff)) * 0.8, 0.01)
        im = axes[2].pcolormesh(phase_centers, freqs, diff,
                                cmap='RdBu_r', shading='auto', vmin=-vd, vmax=vd)
        plt.colorbar(im, ax=axes[2], label='$\\Delta$PPC')
        axes[2].set_title('Phasic - Tonic')
        axes[2].set_ylabel('Frequency (Hz)')
        axes[2].set_xlabel('Theta phase (deg)')

        plt.tight_layout()
        fname = os.path.join(save_dir, f'Rat{rat_id}_{label}_heatmap.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        saved_files.append(fname)
        plt.close(fig)

    print(f"Saved {len(saved_files)} heatmap figures to: {save_dir}/")
    for f in saved_files:
        print(f"  {f}")


def plot_trial_ppc_curves(trial_results, rat_id, condition=None, freq_range=(15, 140),
                          save_dir='ppc_trial_figures'):
    """
    Plot phase-averaged PPC(f) for each trial as a separate PNG.
    Red = phasic, blue = tonic. All trials guaranteed to have both.
    """
    os.makedirs(save_dir, exist_ok=True)

    trials = [t for t in trial_results if t['rat_id'] == rat_id]
    if condition is not None:
        trials = [t for t in trials if t['condition'] == condition]

    saved_files = []
    for t in trials:
        freqs = t['frequencies']
        mask = (freqs >= freq_range[0]) & (freqs <= freq_range[1])
        label = f"{t['condition']}_SD{t['sd']}_Trial{t['trial_number']}"

        fig, ax = plt.subplots(figsize=(10, 4))

        ppc_ph = np.mean(t['phasic_ppc'], axis=1)
        ax.plot(freqs[mask], ppc_ph[mask], color='red', linewidth=1.5,
                label=f'Phasic (n={t["phasic_n"]})')

        ppc_to = np.mean(t['tonic_ppc'], axis=1)
        ax.plot(freqs[mask], ppc_to[mask], color='blue', linewidth=1.5,
                label=f'Tonic (n={t["tonic_n"]})')

        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('PPC (phase-averaged)')
        ax.set_title(f'Rat {rat_id} — {t["condition"]}/SD{t["sd"]}/Trial{t["trial_number"]}')
        ax.legend()
        sns.despine()
        plt.tight_layout()

        fname = os.path.join(save_dir, f'Rat{rat_id}_{label}_curve.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        saved_files.append(fname)
        plt.close(fig)

    print(f"Saved {len(saved_files)} curve figures to: {save_dir}/")
    for f in saved_files:
        print(f"  {f}")

In [ ]:
# ---- Browse a specific rat's trials ----
# Heatmaps: each row = one trial, columns = phasic / tonic / difference
plot_trial_ppc_heatmaps(trial_results, rat_id=9)  # condition='HomeCageHC' to filter

In [ ]:
# ---- Overlay all trial PPC(f) curves for one rat ----
plot_trial_ppc_curves(trial_results, rat_id=9)  # condition='HomeCageHC' to filter

### Trial-averaged PPC
Average PPC(f, φ) across trials. Each trial contributes equally (unlike pooled PPC where trials with more cycles dominate). Optional `min_cycles` threshold to exclude noisy trials.

In [ ]:
def average_trial_ppc(trial_results, rat_id=None, condition=None, min_cycles=10,
                      exclude=None):
    """
    Average PPC(f, phi) across trials. All trials already have both phasic & tonic.
    Additional min_cycles filter can further exclude low-cycle trials.

    Parameters
    ----------
    trial_results : list of dict
        Output of compute_per_trial_ppc (only trials with both states).
    rat_id : int or None
        If int, average only that rat's trials. If None, average across all rats.
    condition : str or None
        Filter by condition folder name. None = all conditions.
    min_cycles : int
        Minimum cycles required (applied to BOTH phasic and tonic).
    exclude : list of int or list of tuple, optional
        Trials to exclude. Can be:
          - list of int: indices into trial_results (from the summary table)
          - list of (rat_id, condition, sd, trial_number) tuples for precise matching

    Returns
    -------
    result : dict
    """
    # Build exclusion set
    if exclude is None:
        exclude_set = set()
        exclude_indices = set()
    elif exclude and isinstance(exclude[0], int):
        exclude_indices = set(exclude)
        exclude_set = set()
    else:
        exclude_indices = set()
        exclude_set = set(tuple(e) for e in exclude)

    # Filter with exclusion
    trials = []
    for idx, t in enumerate(trial_results):
        if idx in exclude_indices:
            continue
        key = (t['rat_id'], t['condition'], t['sd'], t['trial_number'])
        if key in exclude_set:
            continue
        trials.append(t)

    if rat_id is not None:
        trials = [t for t in trials if t['rat_id'] == rat_id]
    if condition is not None:
        trials = [t for t in trials if t['condition'] == condition]

    # Filter: both phasic and tonic must have >= min_cycles
    trials = [t for t in trials
              if t['phasic_n'] >= min_cycles and t['tonic_n'] >= min_cycles]

    rat_ids_present = sorted(set(t['rat_id'] for t in trials)) if trials else []

    n_excluded = len(exclude_indices) + len(exclude_set)
    if n_excluded > 0:
        print(f"  Excluded {n_excluded} trial(s) from averaging.")

    result = {
        'frequencies': trials[0]['frequencies'] if trials else None,
        'n_phase_bins': trials[0]['n_phase_bins'] if trials else None,
        'n_trials': len(trials),
    }

    for label in ['phasic', 'tonic']:
        all_ppcs = [t[f'{label}_ppc'] for t in trials]

        if all_ppcs:
            arr = np.array(all_ppcs)
            result[f'{label}_avg'] = np.mean(arr, axis=0)
            result[f'{label}_sem'] = np.std(arr, axis=0) / np.sqrt(len(arr))
            result[f'{label}_n_trials'] = len(arr)
        else:
            result[f'{label}_avg'] = None
            result[f'{label}_sem'] = None
            result[f'{label}_n_trials'] = 0

        # Per-rat averages
        per_rat = {}
        for rid in rat_ids_present:
            rat_ppcs = [t[f'{label}_ppc'] for t in trials if t['rat_id'] == rid]
            per_rat[rid] = np.mean(rat_ppcs, axis=0) if rat_ppcs else None
        result[f'{label}_per_rat'] = per_rat

    return result


def plot_trial_averaged_ppc_heatmap(avg_result, title='Trial-Averaged PPC',
                                     vmin=-0.05, vmax=0.15, save_path=None):
    """Plot trial-averaged PPC(f, phi) heatmaps: phasic, tonic, difference."""
    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
    freqs = avg_result['frequencies']
    n_tr = avg_result['n_trials']

    ppc_ph = avg_result['phasic_avg']
    ppc_to = avg_result['tonic_avg']

    if ppc_ph is None or ppc_to is None:
        print("Not enough trials with both phasic & tonic to plot.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{title} ({n_tr} trials)', fontsize=14, fontweight='bold')

    im = axes[0].pcolormesh(phase_centers, freqs, ppc_ph,
                             cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=axes[0], label='PPC')
    axes[0].set_title(f'Phasic ({avg_result["phasic_n_trials"]} trials)')
    axes[0].set_ylabel('Frequency (Hz)')
    axes[0].set_xlabel('Theta phase (deg)')

    im = axes[1].pcolormesh(phase_centers, freqs, ppc_to,
                             cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=axes[1], label='PPC')
    axes[1].set_title(f'Tonic ({avg_result["tonic_n_trials"]} trials)')
    axes[1].set_ylabel('Frequency (Hz)')
    axes[1].set_xlabel('Theta phase (deg)')

    diff = ppc_ph - ppc_to
    vd = max(np.max(np.abs(diff)) * 0.8, 0.01)
    im = axes[2].pcolormesh(phase_centers, freqs, diff,
                             cmap='RdBu_r', shading='auto', vmin=-vd, vmax=vd)
    plt.colorbar(im, ax=axes[2], label='$\\Delta$PPC')
    axes[2].set_title('Phasic - Tonic')
    axes[2].set_ylabel('Frequency (Hz)')
    axes[2].set_xlabel('Theta phase (deg)')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()


def plot_trial_averaged_ppc_curves(avg_result, title='Trial-Averaged PPC(f)',
                                    freq_range=(15, 140), save_path=None):
    """Plot trial-averaged phase-averaged PPC(f) with SEM shading."""
    freqs = avg_result['frequencies']
    if freqs is None:
        print("No data to plot.")
        return
    mask = (freqs >= freq_range[0]) & (freqs <= freq_range[1])

    fig, ax = plt.subplots(figsize=(10, 5))

    for label, color in [('phasic', 'red'), ('tonic', 'blue')]:
        avg = avg_result[f'{label}_avg']
        sem = avg_result[f'{label}_sem']
        n = avg_result[f'{label}_n_trials']
        if avg is not None:
            ppc_f = np.mean(avg, axis=1)
            sem_f = np.mean(sem, axis=1)
            ax.plot(freqs[mask], ppc_f[mask], color=color, linewidth=2,
                    label=f'{label.capitalize()} ({n} trials)')
            ax.fill_between(freqs[mask], ppc_f[mask] - sem_f[mask], ppc_f[mask] + sem_f[mask],
                            color=color, alpha=0.2)

    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PPC (phase-averaged)')
    ax.set_title(title)
    ax.legend()
    sns.despine()
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()


def plot_trial_averaged_per_rat(trial_results, rat_ids=None, min_cycles=10,
                                 vmin=-0.05, vmax=0.15, save_dir=None,
                                 exclude=None):
    """
    Plot trial-averaged PPC for each rat separately, then grand average.
    All trials already guaranteed to have both phasic & tonic.

    Parameters
    ----------
    exclude : list of int or list of tuple, optional
        Trials to exclude (passed through to average_trial_ppc).
        - list of int: indices into trial_results (from the summary table)
        - list of (rat_id, condition, sd, trial_number) tuples
    """
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    if rat_ids is None:
        rat_ids = sorted(set(t['rat_id'] for t in trial_results))

    # Per-rat plots
    for rid in rat_ids:
        avg = average_trial_ppc(trial_results, rat_id=rid, min_cycles=min_cycles,
                                exclude=exclude)
        if avg['n_trials'] == 0:
            print(f"Rat {rid}: no trials with both states >= {min_cycles} cycles")
            continue

        sp_hm = os.path.join(save_dir, f'Rat{rid}_trial_avg_heatmap.png') if save_dir else None
        sp_cv = os.path.join(save_dir, f'Rat{rid}_trial_avg_curve.png') if save_dir else None

        plot_trial_averaged_ppc_heatmap(
            avg, title=f'Rat {rid} — Trial-Averaged PPC (min_cycles={min_cycles})',
            vmin=vmin, vmax=vmax, save_path=sp_hm)
        plot_trial_averaged_ppc_curves(
            avg, title=f'Rat {rid} — Trial-Averaged PPC(f) (min_cycles={min_cycles})',
            save_path=sp_cv)

    # Grand average across all selected rats
    if len(rat_ids) > 1:
        trials_filtered = [t for t in trial_results if t['rat_id'] in rat_ids]
        avg_all = average_trial_ppc(trials_filtered, rat_id=None, min_cycles=min_cycles,
                                    exclude=exclude)

        if avg_all['n_trials'] == 0:
            print(f"No trials across rats with both states >= {min_cycles} cycles")
            return

        sp_hm = os.path.join(save_dir, 'GrandAvg_trial_avg_heatmap.png') if save_dir else None
        sp_cv = os.path.join(save_dir, 'GrandAvg_trial_avg_curve.png') if save_dir else None

        plot_trial_averaged_ppc_heatmap(
            avg_all,
            title=f'Grand Average — Trial-Averaged PPC (n={len(rat_ids)} rats, min_cycles={min_cycles})',
            vmin=vmin, vmax=vmax, save_path=sp_hm)
        plot_trial_averaged_ppc_curves(
            avg_all,
            title=f'Grand Average — Trial-Averaged PPC(f) (n={len(rat_ids)} rats, min_cycles={min_cycles})',
            save_path=sp_cv)

        # Per-rat overlay figure
        freqs = avg_all['frequencies']
        fmask = (freqs >= 15) & (freqs <= 140)

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle(f'Per-Rat Trial-Averaged PPC(f) (min_cycles={min_cycles})',
                     fontsize=14, fontweight='bold')

        for label, color, ax in [('phasic', 'red', axes[0]), ('tonic', 'blue', axes[1])]:
            per_rat = avg_all[f'{label}_per_rat']
            curves = []
            for rid in sorted(per_rat.keys()):
                if per_rat[rid] is not None:
                    ppc_f = np.mean(per_rat[rid], axis=1)
                    ax.plot(freqs[fmask], ppc_f[fmask], alpha=0.5, linewidth=1,
                            label=f'Rat {rid}')
                    curves.append(ppc_f[fmask])
            if curves:
                arr = np.array(curves)
                mean_c = np.mean(arr, axis=0)
                sem_c = np.std(arr, axis=0) / np.sqrt(len(arr))
                ax.plot(freqs[fmask], mean_c, color='black', linewidth=2.5, label='Mean')
                ax.fill_between(freqs[fmask], mean_c - sem_c, mean_c + sem_c,
                                color='gray', alpha=0.3)
            ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
            ax.set_xlabel('Frequency (Hz)')
            ax.set_ylabel('PPC')
            ax.set_title(f'{label.capitalize()} REM')
            ax.legend(fontsize=7, loc='upper right')
            sns.despine(ax=ax)

        plt.tight_layout()
        if save_dir:
            sp = os.path.join(save_dir, 'PerRat_overlay_curves.png')
            fig.savefig(sp, dpi=150, bbox_inches='tight')
            print(f"Saved to {sp}")
        plt.show()

In [ ]:
# ---- Trial-averaged PPC for a single rat ----

# Exclude specific bad trials by index (row number from the summary table above).
# Look at the summary table / individual trial plots to identify outliers.
# Example: exclude=[3, 7, 12]   — removes rows 3, 7, 12 from averaging
# Or by identity: exclude=[(9, 'HomeCageHC', '4', 1)]  — removes that specific trial
exclude_trials = []  # <-- ADD BAD TRIAL INDICES HERE

plot_trial_averaged_per_rat(
    trial_results,
    rat_ids=[1],          # <-- single rat
    min_cycles=10,
    save_dir='ppc_trial_avg_figures',
    exclude=exclude_trials
)

In [ ]:
exclude_trials = [
    (1, 'OverlappingOR', '3', 1),
    (3, 'RandomCon', '5', 1),
    (3, 'RandomCon', '5', 2),
    (3, 'StableCondOD', '1', 5),
    (9, 'StableCondOD', '6', 2)
]

In [ ]:
# ---- Trial-averaged PPC across all rats (grand average) ----
# Change rat_ids to match what you ran in compute_per_trial_ppc
plot_trial_averaged_per_rat(
    trial_results,
    rat_ids=[1, 3, 6, 9],   # <-- all rats; shows per-rat + grand avg
    min_cycles=10,
    save_dir='ppc_trial_avg_figures',
    exclude=exclude_trials   # <-- same exclusion list from above
)

### Positive rats

In [ ]:
# ---- Run per-trial PPC ----
selected_rats_trials = [2, 4, 7, 8]  # <-- CHANGE THIS

trial_results = compute_per_trial_ppc(
    rat_ids=selected_rats_trials,
    save_path='ppc_per_trial_results.pkl'
)

In [ ]:
# Quick summary table of all trials
summary_rows = []
for t in trial_results:
    summary_rows.append({
        'Rat': t['rat_id'],
        'Condition': t['condition'],
        'SD': t['sd'],
        'Trial': t['trial_number'],
        'Folder': t['folder'],
        'Phasic cycles': t['phasic_n'],
        'Tonic cycles': t['tonic_n'],
        'Has phasic PPC': t['phasic_ppc'] is not None,
        'Has tonic PPC': t['tonic_ppc'] is not None,
    })
trial_summary = pd.DataFrame(summary_rows)
trial_summary

In [ ]:
# ---- Browse a specific rat's trials ----
# Heatmaps: each row = one trial, columns = phasic / tonic / difference
plot_trial_ppc_heatmaps(trial_results, rat_id=8, save_dir='figures/ppc_trial_positive_rats_figures')

  # condition='HomeCageHC' to filter

In [ ]:
# ---- Overlay all trial PPC(f) curves for one rat ----
plot_trial_ppc_curves(trial_results, rat_id=2, save_dir='figures/ppc_trial_positive_rats_figures')  # condition='HomeCageHC' to filter

In [ ]:
exclude_trials = [
    (2, 'HomeCageHC', '4', 1),
    (3, 'RandomCon', '2', 3),
]

In [ ]:
# ---- Trial-averaged PPC across all rats (grand average) ----
# Change rat_ids to match what you ran in compute_per_trial_ppc
plot_trial_averaged_per_rat(
    trial_results,
    rat_ids=[2, 4, 7, 8],   # <-- all rats you computed; shows per-rat + grand avg
    min_cycles=10,
    save_dir='figures/ppc_trial_avg_positive_rats_figures',
    exclude=exclude_trials
)

# Full Dataset PPC Pipeline
# Field-Field PPC across all rats, conditions, and post-trials
# Following Zhang et al. (2019) - eLife

In [ ]:
def find_file(directory, prefix):
    """Auto-discover a .mat file by prefix (e.g. 'HPC_100', 'PFC_100', 'states')."""
    for fname in os.listdir(directory):
        low = fname.lower()
        if low.startswith(prefix.lower()) and low.endswith('.mat'):
            return os.path.join(directory, fname)
        # states file has a different pattern
        if prefix == 'states' and 'states' in low and low.endswith('.mat'):
            return os.path.join(directory, fname)
    return None


def compute_ppc_for_dataset(rat_ids='all', base_path='/Users/amir/Desktop/for Abdel/RGS/DatabyCondition',
                            fs=1000, frequencies=np.arange(15, 141, 1),
                            n_phase_bins=20, n_cycles_wavelet=5,
                            save_path=None):
    """
    Full dataset PPC pipeline: traverse all rats/conditions/post-trials,
    compute field-field PPC between HPC and PFC for phasic and tonic REM.

    Parameters
    ----------
    rat_ids : list of int or 'all'
        Which rats to process. E.g. [1, 2, 3] or 'all'.
    base_path : str
        Path to the DatabyCondition folder.
    fs : int
        Sampling rate.
    frequencies : ndarray
        Frequencies for wavelet transform.
    n_phase_bins : int
        Number of theta phase bins per cycle.
    n_cycles_wavelet : int
        Morlet wavelet cycles.
    save_path : str or None
        If provided, save results dict as a pickle file.

    Returns
    -------
    results : dict
        Nested dict: results[rat_id] = {
            'phasic_W': list of W_k matrices (all theta cycles, all sessions),
            'tonic_W': list of W_k matrices,
            'phasic_ppc': ndarray (n_freqs, n_phase_bins),
            'tonic_ppc': ndarray,
            'phasic_n': int,
            'tonic_n': int,
            'metadata': list of dicts with session info per cycle
        }
    """

    cfg = globals().get("config", None)
    if cfg is None:
        raise RuntimeError("Define `config` (emd SiftConfig) before calling this function.")

    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    if not conditions:
        print(f"No condition folders found under: {base_path}")
        return None

    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    # --- Discover all available rat IDs if 'all' ---
    if rat_ids == 'all':
        all_rat_ids = set()
        for cond in conditions:
            cond_path = os.path.join(base_path, cond)
            for f in os.listdir(cond_path):
                m = folder_re.match(f)
                if m:
                    all_rat_ids.add(int(m.group(1)))
        rat_ids = sorted(all_rat_ids)
        print(f"Discovered rats: {rat_ids}")

    results = {}

    for rat_id in rat_ids:
        print(f"\n{'='*60}")
        print(f"  Processing Rat {rat_id}")
        print(f"{'='*60}")

        phasic_W_all = []  # all W_k phase-diff matrices for phasic
        tonic_W_all = []   # all W_k phase-diff matrices for tonic
        metadata_phasic = []
        metadata_tonic = []

        for condition_folder in conditions:
            condition_path = os.path.join(base_path, condition_folder)

            rat_folders = []
            for f in os.listdir(condition_path):
                full = os.path.join(condition_path, f)
                if not os.path.isdir(full):
                    continue
                m = folder_re.match(f)
                if m and int(m.group(1)) == rat_id:
                    rat_folders.append((f, m))

            if not rat_folders:
                continue

            for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
                rat_path = os.path.join(condition_path, rat_folder)
                sd_number = m.group(2)
                condition = m.group(3)

                post_trial_folders = sorted([
                    f for f in os.listdir(rat_path)
                    if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
                ])

                for pt_folder in post_trial_folders:
                    trial_path = os.path.join(rat_path, pt_folder)

                    # Auto-discover files
                    hpc_file = find_file(trial_path, 'HPC_100')
                    pfc_file = find_file(trial_path, 'PFC_100')
                    state_file = find_file(trial_path, 'states')

                    if not hpc_file or not pfc_file or not state_file:
                        missing = []
                        if not hpc_file: missing.append('HPC')
                        if not pfc_file: missing.append('PFC')
                        if not state_file: missing.append('states')
                        print(f"  [SKIP] {condition}/{pt_folder} - missing: {', '.join(missing)}")
                        continue

                    trial_match = re.search(r'Post[-_]Trial(\d+)', pt_folder)
                    trial_number = int(trial_match.group(1)) if trial_match else -1

                    session_info = {
                        'rat_id': rat_id, 'condition': condition_folder,
                        'sd': sd_number, 'trial': trial_number, 'folder': pt_folder
                    }

                    try:
                        # Load data
                        lfpHPC_r, hypno_r, _ = get_data(hpc_file, state_file)
                        lfpPFC_r, _, _ = get_data(pfc_file, state_file, type='pfc')

                        try:
                            phasic_int, tonic_int, lfp_raw_r = extract_pt_intervals(
                                lfpHPC_r, hypno_r, fs=fs)
                        except ValueError:
                            print(f"  [SKIP] {condition_folder}/{pt_folder} - no REM sleep")
                            continue

                        has_phasic = phasic_int is not None and len(phasic_int) > 0
                        has_tonic = tonic_int is not None and len(tonic_int) > 0

                        if not has_phasic and not has_tonic:
                            print(f"  [SKIP] {condition_folder}/{pt_folder} - no phasic/tonic intervals")
                            continue

                        # --- Process PHASIC ---
                        if has_phasic:
                            try:
                                ph_imfs, ph_freqs, _ = extract_imfs_by_pt_intervals(
                                    lfp_raw_r, fs, phasic_int, cfg, return_imfs_freqs=True)
                                pfc_ph_lfp = extract_lfp_by_pt_intervals(lfpPFC_r, fs, phasic_int)

                                W_list = _extract_ppc_phase_diffs(
                                    ph_imfs, pfc_ph_lfp, fs, frequencies, n_phase_bins, n_cycles_wavelet)
                                phasic_W_all.extend(W_list)
                                metadata_phasic.extend([session_info] * len(W_list))
                            except Exception as e:
                                print(f"  [WARN] Phasic failed {condition_folder}/{pt_folder}: {e}")

                        # --- Process TONIC ---
                        if has_tonic:
                            try:
                                to_imfs, to_freqs, _ = extract_imfs_by_pt_intervals(
                                    lfp_raw_r, fs, tonic_int, cfg, return_imfs_freqs=True)
                                pfc_to_lfp = extract_lfp_by_pt_intervals(lfpPFC_r, fs, tonic_int)

                                W_list = _extract_ppc_phase_diffs(
                                    to_imfs, pfc_to_lfp, fs, frequencies, n_phase_bins, n_cycles_wavelet)
                                tonic_W_all.extend(W_list)
                                metadata_tonic.extend([session_info] * len(W_list))
                            except Exception as e:
                                print(f"  [WARN] Tonic failed {condition_folder}/{pt_folder}: {e}")

                        n_ph = len(phasic_W_all)
                        n_to = len(tonic_W_all)
                        print(f"  [OK]   {condition_folder}/{pt_folder} | "
                              f"phasic cycles so far: {n_ph}, tonic: {n_to}")

                    except Exception as e:
                        print(f"  [ERROR] {condition_folder}/{pt_folder}: {e}")
                        continue

        # --- Compute PPC from collected phase differences ---
        rat_result = {
            'frequencies': frequencies,
            'n_phase_bins': n_phase_bins,
            'metadata_phasic': metadata_phasic,
            'metadata_tonic': metadata_tonic,
        }

        for label, W_all in [('phasic', phasic_W_all), ('tonic', tonic_W_all)]:
            N = len(W_all)
            rat_result[f'{label}_n'] = N
            if N >= 2:
                sum_vec = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
                for W_k in W_all:
                    sum_vec += np.exp(1j * W_k)
                ppc = (np.abs(sum_vec)**2 - N) / (N * (N - 1))
                rat_result[f'{label}_ppc'] = ppc
            else:
                rat_result[f'{label}_ppc'] = None
                print(f"  Rat {rat_id} {label}: only {N} cycles, cannot compute PPC")

        print(f"\n  Rat {rat_id} summary: phasic={rat_result['phasic_n']} cycles, "
              f"tonic={rat_result['tonic_n']} cycles")
        results[rat_id] = rat_result

    # Optionally save
    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"\nResults saved to {save_path}")

    return results


def _extract_ppc_phase_diffs(hpc_imfs, pfc_lfp_segments, fs, frequencies,
                              n_phase_bins, n_cycles_wavelet):
    """
    Extract per-cycle phase-difference matrices W_k for PPC computation.
    Same logic as compute_field_field_ppc but returns the raw W_k list
    so they can be accumulated across sessions.
    """
    all_W = []

    for idx in range(len(hpc_imfs)):
        imf = hpc_imfs[idx]
        pfc = pfc_lfp_segments[idx]

        hpc = np.sum(imf, axis=1)
        min_len = min(len(hpc), len(pfc))
        hpc, pfc = hpc[:min_len], pfc[:min_len]
        theta_imf = imf[:min_len, 5]

        # Detect theta cycles
        cycle_data = get_cycle_data(theta_imf, fs)
        amp_thresh = np.percentile(cycle_data['IA'], 25)
        conditions = [
            'is_good==1',
            f'duration_samples<{fs / 5}',
            f'duration_samples>{fs / 12}',
            f'max_amp>{amp_thresh}'
        ]
        all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
        if all_cycles is None or all_cycles.chain_vect.size == 0:
            continue

        subset_df = all_cycles.get_metric_dataframe(subset=True)
        cycle_inds = arrange_cycle_inds(
            get_cycle_inds(all_cycles, subset_df['index'].values)
        )

        # Complex wavelet transforms
        hpc_cwt = sails.wavelet.morlet(
            hpc, freqs=frequencies, sample_rate=fs,
            ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
        )
        pfc_cwt = sails.wavelet.morlet(
            pfc, freqs=frequencies, sample_rate=fs,
            ncycles=n_cycles_wavelet, ret_mode='complex', normalise='simple'
        )

        cross = hpc_cwt * np.conj(pfc_cwt)

        for c in range(len(cycle_inds)):
            start, end = cycle_inds[c]
            cycle_len = end - start
            if end > min_len or cycle_len < n_phase_bins:
                continue

            cs_cycle = cross[:, start:end]
            bin_edges = np.linspace(0, cycle_len, n_phase_bins + 1, dtype=int)
            B_k = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
            for pb in range(n_phase_bins):
                b_s, b_e = bin_edges[pb], bin_edges[pb + 1]
                if b_e > b_s:
                    B_k[:, pb] = np.mean(cs_cycle[:, b_s:b_e], axis=1)

            all_W.append(np.angle(B_k))

    return all_W

In [ ]:
# ---- Choose which rats to run ----
# Options: 'all', or a list like [1, 2, 3]
selected_rats = [1, 3, 6, 9]  # <-- CHANGE THIS to select rats

results = compute_ppc_for_dataset(
    rat_ids=selected_rats,
    save_path='ppc_results.pkl'  # saves a pickle for later use
)

## Per-rat PPC heatmaps (Phasic vs Tonic)

In [ ]:
def plot_ppc_per_rat(results, vmin=-0.1, vmax=0.1):
    """Plot PPC heatmaps (phasic, tonic, difference) for each rat."""
    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

    for rat_id, res in sorted(results.items()):
        freqs = res['frequencies']
        ppc_ph = res['phasic_ppc']
        ppc_to = res['tonic_ppc']
        n_ph = res['phasic_n']
        n_to = res['tonic_n']

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Rat {rat_id} - HPC-PFC Field-Field PPC', fontsize=14, fontweight='bold')

        if ppc_ph is not None:
            im0 = axes[0].pcolormesh(phase_centers, freqs, ppc_ph,
                                      cmap='hot', shading='auto', vmin=vmin, vmax=vmax)
            axes[0].set_title(f'Phasic REM (n={n_ph})')
            axes[0].set_xlabel('Theta phase (deg)')
            axes[0].set_ylabel('Frequency (Hz)')
            plt.colorbar(im0, ax=axes[0], label='PPC')
        else:
            axes[0].set_title(f'Phasic REM (n={n_ph}) - insufficient data')

        if ppc_to is not None:
            im1 = axes[1].pcolormesh(phase_centers, freqs, ppc_to,
                                      cmap='hot', shading='auto', vmin=vmin, vmax=vmax)
            axes[1].set_title(f'Tonic REM (n={n_to})')
            axes[1].set_xlabel('Theta phase (deg)')
            axes[1].set_ylabel('Frequency (Hz)')
            plt.colorbar(im1, ax=axes[1], label='PPC')
        else:
            axes[1].set_title(f'Tonic REM (n={n_to}) - insufficient data')

        if ppc_ph is not None and ppc_to is not None:
            ppc_diff = ppc_ph - ppc_to
            vmax_d = np.max(np.abs(ppc_diff)) * 0.8
            im2 = axes[2].pcolormesh(phase_centers, freqs, ppc_diff,
                                      cmap='hot', shading='auto', vmin=-vmax_d, vmax=vmax_d)
            axes[2].set_title('Phasic - Tonic')
            axes[2].set_xlabel('Theta phase (deg)')
            axes[2].set_ylabel('Frequency (Hz)')
            plt.colorbar(im2, ax=axes[2], label='$\Delta$PPC')
        else:
            axes[2].set_title('Difference - insufficient data')

        plt.tight_layout()
        plt.show()


plot_ppc_per_rat(results)

## Phase-averaged PPC(f) across rats

In [ ]:
def plot_ppc_frequency_summary(results):
    """
    Plot phase-averaged PPC(f) for all rats.
    Left: individual rats overlaid. Right: grand average +/- SEM.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    phasic_curves = []
    tonic_curves = []
    freqs = None

    # --- Left panel: individual rats ---
    for rat_id, res in sorted(results.items()):
        freqs = res['frequencies']

        if res['phasic_ppc'] is not None:
            ph_avg = np.mean(res['phasic_ppc'], axis=1)
            axes[0].plot(freqs, ph_avg, color='red', alpha=0.4, linewidth=1,
                         label=f'Phasic Rat{rat_id}')
            phasic_curves.append(ph_avg)

        if res['tonic_ppc'] is not None:
            to_avg = np.mean(res['tonic_ppc'], axis=1)
            axes[0].plot(freqs, to_avg, color='blue', alpha=0.4, linewidth=1,
                         label=f'Tonic Rat{rat_id}')
            tonic_curves.append(to_avg)

    axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Frequency (Hz)')
    axes[0].set_ylabel('PPC (phase-averaged)')
    axes[0].set_title('Individual Rats')
    # Only show legend if few rats
    if len(results) <= 6:
        axes[0].legend(fontsize=7, loc='upper right')
    sns.despine(ax=axes[0])

    # --- Right panel: grand average +/- SEM ---
    if phasic_curves:
        ph_arr = np.array(phasic_curves)
        ph_mean = np.mean(ph_arr, axis=0)
        ph_sem = np.std(ph_arr, axis=0) / np.sqrt(len(ph_arr))
        axes[1].plot(freqs, ph_mean, color='red', linewidth=2, label=f'Phasic (n={len(ph_arr)} rats)')
        axes[1].fill_between(freqs, ph_mean - ph_sem, ph_mean + ph_sem, color='red', alpha=0.2)

    if tonic_curves:
        to_arr = np.array(tonic_curves)
        to_mean = np.mean(to_arr, axis=0)
        to_sem = np.std(to_arr, axis=0) / np.sqrt(len(to_arr))
        axes[1].plot(freqs, to_mean, color='blue', linewidth=2, label=f'Tonic (n={len(to_arr)} rats)')
        axes[1].fill_between(freqs, to_mean - to_sem, to_mean + to_sem, color='blue', alpha=0.2)

    axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel('PPC (phase-averaged)')
    axes[1].set_title('Grand Average (+/- SEM)')
    axes[1].legend()
    sns.despine(ax=axes[1])

    plt.tight_layout()
    plt.show()


plot_ppc_frequency_summary(results)

## Grand-average PPC heatmap (across all rats)

In [ ]:
def plot_grand_average_ppc(results, vmin=-0.05, vmax=0.15):
    """Grand-average PPC heatmaps across all rats (mean of per-rat PPC matrices)."""
    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

    phasic_ppcs, tonic_ppcs = [], []
    freqs = None

    for rat_id, res in results.items():
        freqs = res['frequencies']
        if res['phasic_ppc'] is not None:
            phasic_ppcs.append(res['phasic_ppc'])
        if res['tonic_ppc'] is not None:
            tonic_ppcs.append(res['tonic_ppc'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Grand Average HPC-PFC PPC (n={len(results)} rats)', fontsize=14, fontweight='bold')

    if phasic_ppcs:
        ga_ph = np.mean(phasic_ppcs, axis=0)
        im0 = axes[0].pcolormesh(phase_centers, freqs, ga_ph,
                                  cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        axes[0].set_title(f'Phasic REM (n={len(phasic_ppcs)} rats)')
        axes[0].set_xlabel('Theta phase (deg)')
        axes[0].set_ylabel('Frequency (Hz)')
        plt.colorbar(im0, ax=axes[0], label='PPC')

    if tonic_ppcs:
        ga_to = np.mean(tonic_ppcs, axis=0)
        im1 = axes[1].pcolormesh(phase_centers, freqs, ga_to,
                                  cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        axes[1].set_title(f'Tonic REM (n={len(tonic_ppcs)} rats)')
        axes[1].set_xlabel('Theta phase (deg)')
        axes[1].set_ylabel('Frequency (Hz)')
        plt.colorbar(im1, ax=axes[1], label='PPC')

    if phasic_ppcs and tonic_ppcs:
        ga_diff = ga_ph - ga_to
        vmax_d = np.max(np.abs(ga_diff)) * 0.8
        im2 = axes[2].pcolormesh(phase_centers, freqs, ga_diff,
                                  cmap='RdBu_r', shading='auto', vmin=-vmax_d, vmax=vmax_d)
        axes[2].set_title('Phasic - Tonic')
        axes[2].set_xlabel('Theta phase (deg)')
        axes[2].set_ylabel('Frequency (Hz)')
        plt.colorbar(im2, ax=axes[2], label='$\Delta$PPC')

    plt.tight_layout()
    plt.show()


plot_grand_average_ppc(results)

## Load saved results (optional)
Use this to reload previously computed results without re-running the pipeline.

In [ ]:
# # Reload saved results
# with open('ppc_results.pkl', 'rb') as f:
#     results = pickle.load(f)
# print(f"Loaded results for rats: {list(results.keys())}")
# for rat_id, res in results.items():
#     print(f"  Rat {rat_id}: phasic={res['phasic_n']} cycles, tonic={res['tonic_n']} cycles")

# Field-Field PPC using `spectral_connectivity` package
# Validation using Multitaper + PPC from Eden-Kramer-Lab
# Each theta cycle = one trial; PPC computed across cycles at each frequency

In [ ]:
from spectral_connectivity import Multitaper, Connectivity
from scipy.interpolate import interp1d


def extract_cycle_segments(hpc_imfs, pfc_lfp_segments, fs=1000, fixed_n_samples=125):
    """
    Extract HPC and PFC LFP segments for each valid theta cycle.
    Each cycle is resampled to fixed_n_samples via linear interpolation.

    Parameters
    ----------
    hpc_imfs : list of ndarray
        HPC IMFs per REM interval.
    pfc_lfp_segments : list of ndarray
        PFC raw LFP per REM interval.
    fs : int
        Sampling frequency.
    fixed_n_samples : int
        Resample each cycle to this many samples. Default 125 (~8Hz theta).

    Returns
    -------
    hpc_cycles : ndarray, shape (fixed_n_samples, n_cycles)
    pfc_cycles : ndarray, shape (fixed_n_samples, n_cycles)
    n_cycles : int
    """
    hpc_raw_cycles = []
    pfc_raw_cycles = []

    for idx in range(len(hpc_imfs)):
        imf = hpc_imfs[idx]
        pfc = pfc_lfp_segments[idx]

        hpc = np.sum(imf, axis=1)
        min_len = min(len(hpc), len(pfc))
        hpc, pfc = hpc[:min_len], pfc[:min_len]
        theta_imf = imf[:min_len, 5]

        cycle_data = get_cycle_data(theta_imf, fs)
        amp_thresh = np.percentile(cycle_data['IA'], 25)
        conditions = [
            'is_good==1',
            f'duration_samples<{fs / 5}',
            f'duration_samples>{fs / 12}',
            f'max_amp>{amp_thresh}'
        ]
        all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
        if all_cycles is None or all_cycles.chain_vect.size == 0:
            continue

        subset_df = all_cycles.get_metric_dataframe(subset=True)
        cycle_inds = arrange_cycle_inds(
            get_cycle_inds(all_cycles, subset_df['index'].values)
        )

        for c in range(len(cycle_inds)):
            start, end = cycle_inds[c]
            if end > min_len or (end - start) < 10:
                continue
            hpc_raw_cycles.append(hpc[start:end])
            pfc_raw_cycles.append(pfc[start:end])

    if len(hpc_raw_cycles) == 0:
        return None, None, 0

    # Resample every cycle to exactly fixed_n_samples
    target_x = np.linspace(0, 1, fixed_n_samples)
    hpc_resampled = np.zeros((fixed_n_samples, len(hpc_raw_cycles)))
    pfc_resampled = np.zeros((fixed_n_samples, len(pfc_raw_cycles)))

    for i in range(len(hpc_raw_cycles)):
        n = len(hpc_raw_cycles[i])
        source_x = np.linspace(0, 1, n)
        hpc_resampled[:, i] = interp1d(source_x, hpc_raw_cycles[i], kind='linear')(target_x)
        pfc_resampled[:, i] = interp1d(source_x, pfc_raw_cycles[i], kind='linear')(target_x)

    return hpc_resampled, pfc_resampled, len(hpc_raw_cycles)


def compute_ppc_spectral_connectivity(hpc_imfs, pfc_lfp_segments, fs=1000,
                                       time_halfbandwidth_product=4,
                                       fixed_n_samples=125):
    """
    Compute field-field PPC between HPC and PFC using the spectral_connectivity package.
    Each theta cycle = one trial. All cycles resampled to fixed_n_samples.
    """
    hpc_cycles, pfc_cycles, n_cycles = extract_cycle_segments(
        hpc_imfs, pfc_lfp_segments, fs=fs, fixed_n_samples=fixed_n_samples
    )

    if n_cycles < 2:
        print(f'Only {n_cycles} cycles - cannot compute PPC.')
        return None, None, n_cycles

    print(f'Stacking {n_cycles} theta cycles, each {hpc_cycles.shape[0]} samples')

    # Shape: (n_samples, n_cycles, 2_signals)
    time_series = np.stack([hpc_cycles, pfc_cycles], axis=-1)

    m = Multitaper(
        time_series,
        sampling_frequency=fs,
        time_halfbandwidth_product=time_halfbandwidth_product,
    )

    c = Connectivity.from_multitaper(m)
    ppc_full = c.pairwise_phase_consistency()
    freqs = c.frequencies

    ppc = np.squeeze(ppc_full)
    if ppc.ndim == 3:
        ppc_hpc_pfc = ppc[:, 0, 1]
    elif ppc.ndim == 4:
        ppc_hpc_pfc = ppc[0, :, 0, 1]
    else:
        raise ValueError(f"Unexpected PPC shape after squeeze: {ppc.shape}")

    print(f'PPC shape: {ppc_hpc_pfc.shape}, freqs shape: {freqs.shape}')

    return ppc_hpc_pfc, freqs, n_cycles

### Single-trial test (same data as above)

In [ ]:
# Debug shapes
sc_ppc_phasic, sc_freqs_phasic, sc_n_phasic = compute_ppc_spectral_connectivity(
    phasic_imfs, pfc_phasic_rem_lfp, fs=fs
)
print(f'Phasic: {sc_n_phasic} cycles')
print(f'freqs shape: {sc_freqs_phasic.shape}, ppc shape: {sc_ppc_phasic.shape}')

sc_ppc_tonic, sc_freqs_tonic, sc_n_tonic = compute_ppc_spectral_connectivity(
    tonic_imfs, pfc_tonic_rem_lfp, fs=fs
)
print(f'Tonic: {sc_n_tonic} cycles')
print(f'freqs shape: {sc_freqs_tonic.shape}, ppc shape: {sc_ppc_tonic.shape}')

### Comparison: custom PPC vs spectral_connectivity PPC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Phasic comparison ---
ax = axes[0]
if ppc_phasic is not None:
    custom_avg = np.mean(ppc_phasic, axis=1)
    ax.plot(ppc_freqs, custom_avg, label='Custom (Zhang-style, phase-avg)', color='red', linewidth=1.5)
if sc_ppc_phasic is not None:
    # Limit to 15-140 Hz for comparison
    freq_mask = (sc_freqs_phasic >= 15) & (sc_freqs_phasic <= 140)
    ax.plot(sc_freqs_phasic[freq_mask], sc_ppc_phasic[freq_mask],
            label='spectral_connectivity (multitaper)', color='darkred', linewidth=1.5, linestyle='--')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC')
ax.set_title(f'Phasic REM: Custom vs spectral_connectivity')
ax.set_xlim(15, 140)
ax.legend(fontsize=9)
sns.despine(ax=ax)

# --- Tonic comparison ---
ax = axes[1]
if ppc_tonic is not None:
    custom_avg = np.mean(ppc_tonic, axis=1)
    ax.plot(ppc_freqs, custom_avg, label='Custom (Zhang-style, phase-avg)', color='blue', linewidth=1.5)
if sc_ppc_tonic is not None:
    freq_mask = (sc_freqs_tonic >= 15) & (sc_freqs_tonic <= 140)
    ax.plot(sc_freqs_tonic[freq_mask], sc_ppc_tonic[freq_mask],
            label='spectral_connectivity (multitaper)', color='darkblue', linewidth=1.5, linestyle='--')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PPC')
ax.set_title(f'Tonic REM: Custom vs spectral_connectivity')
ax.set_xlim(15, 140)
ax.legend(fontsize=9)
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

### Full dataset PPC via spectral_connectivity

In [ ]:
def _extract_cycle_segments_fixed(hpc_imfs, pfc_lfp_segments, fs, fixed_n_samples):
    """Extract and resample cycle segments to exactly fixed_n_samples. Self-contained."""
    from scipy.interpolate import interp1d as _interp1d

    hpc_raw, pfc_raw = [], []

    for idx in range(len(hpc_imfs)):
        imf = hpc_imfs[idx]
        pfc = pfc_lfp_segments[idx]
        hpc = np.sum(imf, axis=1)
        min_len = min(len(hpc), len(pfc))
        hpc, pfc = hpc[:min_len], pfc[:min_len]
        theta_imf = imf[:min_len, 5]

        cycle_data = get_cycle_data(theta_imf, fs)
        amp_thresh = np.percentile(cycle_data['IA'], 25)
        conditions = [
            'is_good==1',
            f'duration_samples<{fs / 5}',
            f'duration_samples>{fs / 12}',
            f'max_amp>{amp_thresh}'
        ]
        all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
        if all_cycles is None or all_cycles.chain_vect.size == 0:
            continue

        subset_df = all_cycles.get_metric_dataframe(subset=True)
        cycle_inds = arrange_cycle_inds(
            get_cycle_inds(all_cycles, subset_df['index'].values)
        )

        for c in range(len(cycle_inds)):
            start, end = cycle_inds[c]
            if end > min_len or (end - start) < 10:
                continue
            hpc_raw.append(hpc[start:end])
            pfc_raw.append(pfc[start:end])

    if len(hpc_raw) == 0:
        return None, None, 0

    target_x = np.linspace(0, 1, fixed_n_samples)
    hpc_out = np.zeros((fixed_n_samples, len(hpc_raw)))
    pfc_out = np.zeros((fixed_n_samples, len(pfc_raw)))
    for i in range(len(hpc_raw)):
        src = np.linspace(0, 1, len(hpc_raw[i]))
        hpc_out[:, i] = _interp1d(src, hpc_raw[i], kind='linear')(target_x)
        pfc_out[:, i] = _interp1d(src, pfc_raw[i], kind='linear')(target_x)

    return hpc_out, pfc_out, len(hpc_raw)


def compute_ppc_sc_for_dataset(rat_ids='all',
                                base_path='/Users/amir/Desktop/for Abdel/RGS/DatabyCondition',
                                fs=1000, time_halfbandwidth_product=4,
                                fixed_n_samples=125, save_path=None):
    """
    Full dataset PPC pipeline using spectral_connectivity package.
    Each theta cycle = one trial, resampled to fixed_n_samples=125.
    Uses self-contained _extract_cycle_segments_fixed defined above.
    """
    from spectral_connectivity import Multitaper as _Mt, Connectivity as _Cn

    cfg = globals().get("config", None)
    if cfg is None:
        raise RuntimeError("Define `config` (emd SiftConfig) before calling this function.")

    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    if rat_ids == 'all':
        all_rat_ids = set()
        for cond in conditions:
            for f in os.listdir(os.path.join(base_path, cond)):
                m = folder_re.match(f)
                if m:
                    all_rat_ids.add(int(m.group(1)))
        rat_ids = sorted(all_rat_ids)
        print(f"Discovered rats: {rat_ids}")

    print(f"fixed_n_samples={fixed_n_samples}")
    results = {}

    for rat_id in rat_ids:
        print(f"\n{'='*60}")
        print(f"  Processing Rat {rat_id} (spectral_connectivity)")
        print(f"{'='*60}")

        phasic_hpc_all, phasic_pfc_all = [], []
        tonic_hpc_all, tonic_pfc_all = [], []

        for condition_folder in conditions:
            condition_path = os.path.join(base_path, condition_folder)
            rat_folders = []
            for f in os.listdir(condition_path):
                full = os.path.join(condition_path, f)
                if not os.path.isdir(full):
                    continue
                m = folder_re.match(f)
                if m and int(m.group(1)) == rat_id:
                    rat_folders.append((f, m))

            for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
                rat_path = os.path.join(condition_path, rat_folder)
                post_trial_folders = sorted([
                    f for f in os.listdir(rat_path)
                    if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
                ])

                for pt_folder in post_trial_folders:
                    trial_path = os.path.join(rat_path, pt_folder)
                    hpc_file = find_file(trial_path, 'HPC_100')
                    pfc_file = find_file(trial_path, 'PFC_100')
                    state_file = find_file(trial_path, 'states')
                    if not hpc_file or not pfc_file or not state_file:
                        continue

                    try:
                        lfpHPC_r, hypno_r, _ = get_data(hpc_file, state_file)
                        lfpPFC_r, _, _ = get_data(pfc_file, state_file, type='pfc')
                        try:
                            phasic_int, tonic_int, lfp_raw_r = extract_pt_intervals(
                                lfpHPC_r, hypno_r, fs=fs)
                        except ValueError:
                            continue

                        if phasic_int is not None and len(phasic_int) > 0:
                            try:
                                ph_imfs, _, _ = extract_imfs_by_pt_intervals(
                                    lfp_raw_r, fs, phasic_int, cfg, return_imfs_freqs=True)
                                pfc_ph = extract_lfp_by_pt_intervals(lfpPFC_r, fs, phasic_int)
                                h, p, nc = _extract_cycle_segments_fixed(
                                    ph_imfs, pfc_ph, fs, fixed_n_samples)
                                if h is not None:
                                    phasic_hpc_all.append(h)
                                    phasic_pfc_all.append(p)
                            except Exception as e:
                                print(f"  [WARN] Phasic: {e}")

                        if tonic_int is not None and len(tonic_int) > 0:
                            try:
                                to_imfs, _, _ = extract_imfs_by_pt_intervals(
                                    lfp_raw_r, fs, tonic_int, cfg, return_imfs_freqs=True)
                                pfc_to = extract_lfp_by_pt_intervals(lfpPFC_r, fs, tonic_int)
                                h, p, nc = _extract_cycle_segments_fixed(
                                    to_imfs, pfc_to, fs, fixed_n_samples)
                                if h is not None:
                                    tonic_hpc_all.append(h)
                                    tonic_pfc_all.append(p)
                            except Exception as e:
                                print(f"  [WARN] Tonic: {e}")

                        print(f"  [OK] {condition_folder}/{pt_folder}")
                    except Exception as e:
                        print(f"  [ERROR] {condition_folder}/{pt_folder}: {e}")

        # --- Compute PPC per rat ---
        rat_result = {}
        for label, hpc_list, pfc_list in [('phasic', phasic_hpc_all, phasic_pfc_all),
                                            ('tonic', tonic_hpc_all, tonic_pfc_all)]:
            if not hpc_list:
                rat_result[f'{label}_ppc'] = None
                rat_result[f'{label}_freqs'] = None
                rat_result[f'{label}_n'] = 0
                continue

            shapes = set(h.shape[0] for h in hpc_list)
            print(f"  {label} segment dim-0 sizes: {shapes}")

            all_hpc = np.concatenate(hpc_list, axis=1)
            all_pfc = np.concatenate(pfc_list, axis=1)
            n_cycles = all_hpc.shape[1]
            print(f"  Rat {rat_id} {label}: {n_cycles} cycles, shape={all_hpc.shape}")

            if n_cycles < 2:
                rat_result[f'{label}_ppc'] = None
                rat_result[f'{label}_freqs'] = None
                rat_result[f'{label}_n'] = n_cycles
                continue

            time_series = np.stack([all_hpc, all_pfc], axis=-1)
            m_mt = _Mt(time_series, sampling_frequency=fs,
                       time_halfbandwidth_product=time_halfbandwidth_product)
            c = _Cn.from_multitaper(m_mt)
            ppc_full = c.pairwise_phase_consistency()
            ppc = np.squeeze(ppc_full)
            if ppc.ndim == 3:
                ppc = ppc[:, 0, 1]
            elif ppc.ndim == 4:
                ppc = ppc[0, :, 0, 1]

            rat_result[f'{label}_ppc'] = ppc
            rat_result[f'{label}_freqs'] = c.frequencies
            rat_result[f'{label}_n'] = n_cycles

        print(f"\n  Rat {rat_id} summary: phasic={rat_result.get('phasic_n', 0)}, "
              f"tonic={rat_result.get('tonic_n', 0)}")
        results[rat_id] = rat_result

    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(results, f)
        print(f"\nResults saved to {save_path}")

    return results

In [ ]:
# ---- Run spectral_connectivity PPC on selected rats ----
selected_rats_sc = [1]  # <-- CHANGE THIS

results_sc = compute_ppc_sc_for_dataset(
    rat_ids=selected_rats_sc,
    save_path='ppc_results_sc.pkl'
)

### Plot spectral_connectivity PPC results

In [ ]:
def plot_ppc_sc_results(results_sc, freq_range=(15, 140)):
    """Plot PPC(f) from spectral_connectivity for each rat, plus grand average."""
    n_rats = len(results_sc)
    fig, axes = plt.subplots(1, min(n_rats + 1, 4), figsize=(6 * min(n_rats + 1, 4), 5),
                              squeeze=False)
    axes = axes.ravel()

    phasic_curves, tonic_curves = [], []
    common_freqs = None

    for i, (rat_id, res) in enumerate(sorted(results_sc.items())):
        ax = axes[i] if i < len(axes) - 1 else axes[-1]

        if res['phasic_ppc'] is not None:
            f = res['phasic_freqs']
            mask = (f >= freq_range[0]) & (f <= freq_range[1])
            ax.plot(f[mask], res['phasic_ppc'][mask], color='red', linewidth=1.5,
                    label=f"Phasic (n={res['phasic_n']})")
            phasic_curves.append((f[mask], res['phasic_ppc'][mask]))
            if common_freqs is None:
                common_freqs = f[mask]

        if res['tonic_ppc'] is not None:
            f = res['tonic_freqs']
            mask = (f >= freq_range[0]) & (f <= freq_range[1])
            ax.plot(f[mask], res['tonic_ppc'][mask], color='blue', linewidth=1.5,
                    label=f"Tonic (n={res['tonic_n']})")
            tonic_curves.append((f[mask], res['tonic_ppc'][mask]))

        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('PPC')
        ax.set_title(f'Rat {rat_id}')
        ax.legend(fontsize=8)
        sns.despine(ax=ax)

    # Grand average in last panel (if multiple rats)
    if n_rats > 1 and len(axes) > n_rats:
        ax = axes[-1]
        if phasic_curves:
            ph_vals = np.array([c[1] for c in phasic_curves])
            ph_mean = np.mean(ph_vals, axis=0)
            ph_sem = np.std(ph_vals, axis=0) / np.sqrt(len(ph_vals))
            ax.plot(common_freqs, ph_mean, color='red', linewidth=2, label='Phasic')
            ax.fill_between(common_freqs, ph_mean - ph_sem, ph_mean + ph_sem, color='red', alpha=0.2)
        if tonic_curves:
            to_vals = np.array([c[1] for c in tonic_curves])
            to_mean = np.mean(to_vals, axis=0)
            to_sem = np.std(to_vals, axis=0) / np.sqrt(len(to_vals))
            ax.plot(common_freqs, to_mean, color='blue', linewidth=2, label='Tonic')
            ax.fill_between(common_freqs, to_mean - to_sem, to_mean + to_sem, color='blue', alpha=0.2)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('PPC')
        ax.set_title('Grand Average')
        ax.legend()
        sns.despine(ax=ax)

    plt.suptitle('HPC-PFC PPC (spectral_connectivity, multitaper)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_ppc_sc_results(results_sc)

In [ ]:
# ---- Run spectral_connectivity PPC on selected rats ----
selected_rats_sc = [1, 3, 6, 9]  # <-- CHANGE THIS

results_sc = compute_ppc_sc_for_dataset(
    rat_ids=selected_rats_sc,
    save_path='ppc_results_sc.pkl'
)

In [ ]:
plot_ppc_sc_results(results_sc)

## Grand-average PPC (spectral_connectivity, across all rats)

In [ ]:
def plot_grand_average_ppc_sc(results_sc, freq_range=(15, 140)):
    """
    Grand-average PPC(f) across all rats from spectral_connectivity results.
    Left: individual rats overlaid.  Right: grand average +/- SEM.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    phasic_curves, tonic_curves = [], []
    common_freqs = None

    # --- Left: individual rats ---
    for rat_id, res in sorted(results_sc.items()):
        if res['phasic_ppc'] is not None:
            f = res['phasic_freqs']
            mask = (f >= freq_range[0]) & (f <= freq_range[1])
            axes[0].plot(f[mask], res['phasic_ppc'][mask], color='red', alpha=0.35,
                         linewidth=1, label=f'Phasic Rat{rat_id}')
            phasic_curves.append(res['phasic_ppc'][mask])
            if common_freqs is None:
                common_freqs = f[mask]

        if res['tonic_ppc'] is not None:
            f = res['tonic_freqs']
            mask = (f >= freq_range[0]) & (f <= freq_range[1])
            axes[0].plot(f[mask], res['tonic_ppc'][mask], color='blue', alpha=0.35,
                         linewidth=1, label=f'Tonic Rat{rat_id}')
            tonic_curves.append(res['tonic_ppc'][mask])

    axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Frequency (Hz)')
    axes[0].set_ylabel('PPC')
    axes[0].set_title('Individual Rats')
    if len(results_sc) <= 6:
        axes[0].legend(fontsize=7, loc='upper right')
    sns.despine(ax=axes[0])

    # --- Right: grand average +/- SEM ---
    if phasic_curves and common_freqs is not None:
        ph_arr = np.array(phasic_curves)
        ph_mean = np.mean(ph_arr, axis=0)
        ph_sem = np.std(ph_arr, axis=0) / np.sqrt(len(ph_arr))
        axes[1].plot(common_freqs, ph_mean, color='red', linewidth=2,
                     label=f'Phasic (n={len(ph_arr)} rats)')
        axes[1].fill_between(common_freqs, ph_mean - ph_sem, ph_mean + ph_sem,
                             color='red', alpha=0.2)

    if tonic_curves and common_freqs is not None:
        to_arr = np.array(tonic_curves)
        to_mean = np.mean(to_arr, axis=0)
        to_sem = np.std(to_arr, axis=0) / np.sqrt(len(to_arr))
        axes[1].plot(common_freqs, to_mean, color='blue', linewidth=2,
                     label=f'Tonic (n={len(to_arr)} rats)')
        axes[1].fill_between(common_freqs, to_mean - to_sem, to_mean + to_sem,
                             color='blue', alpha=0.2)

    axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel('PPC')
    axes[1].set_title('Grand Average (+/- SEM)')
    axes[1].legend()
    sns.despine(ax=axes[1])

    fig.suptitle('HPC-PFC Grand Average PPC (spectral_connectivity, multitaper)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_grand_average_ppc_sc(results_sc)

## Phase-Frequency PPC heatmaps (wavelet cross-spectrum, same rats)
The `spectral_connectivity` package computes PPC(f) — phase-averaged across the theta cycle.  
To recover PPC(f, φ) as in Zhang et al. (2019), we use the wavelet cross-spectrum approach:  
for each theta cycle, bin the cross-spectrum into theta-phase bins, extract the phase difference, then compute PPC across cycles at each (frequency, phase-bin) pair.

In [ ]:
def compute_phase_freq_ppc_for_dataset(rat_ids='all',
                                       base_path='/Users/amir/Desktop/for Abdel/RGS/DatabyCondition',
                                       fs=1000, frequencies=np.arange(15, 141, 1),
                                       n_phase_bins=20, n_cycles_wavelet=5,
                                       save_path=None):
    """
    Compute phase-frequency PPC(f, phi) using wavelet cross-spectrum.
    Same dataset traversal as the SC pipeline but returns 2D PPC matrices.
    """
    cfg = globals().get("config", None)
    if cfg is None:
        raise RuntimeError("Define `config` before calling.")

    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    if rat_ids == 'all':
        all_rat_ids = set()
        for cond in conditions:
            for f in os.listdir(os.path.join(base_path, cond)):
                m = folder_re.match(f)
                if m:
                    all_rat_ids.add(int(m.group(1)))
        rat_ids = sorted(all_rat_ids)
        print(f"Discovered rats: {rat_ids}")

    results = {}

    for rat_id in rat_ids:
        print(f"\n{'='*60}")
        print(f"  Rat {rat_id} — phase-frequency PPC (wavelet)")
        print(f"{'='*60}")

        phasic_W, tonic_W = [], []

        for condition_folder in conditions:
            condition_path = os.path.join(base_path, condition_folder)
            rat_folders = []
            for f in os.listdir(condition_path):
                if not os.path.isdir(os.path.join(condition_path, f)):
                    continue
                m = folder_re.match(f)
                if m and int(m.group(1)) == rat_id:
                    rat_folders.append((f, m))

            for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
                rat_path = os.path.join(condition_path, rat_folder)
                pt_folders = sorted([
                    f for f in os.listdir(rat_path)
                    if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
                ])

                for pt_folder in pt_folders:
                    trial_path = os.path.join(rat_path, pt_folder)
                    hpc_file = find_file(trial_path, 'HPC_100')
                    pfc_file = find_file(trial_path, 'PFC_100')
                    state_file = find_file(trial_path, 'states')
                    if not hpc_file or not pfc_file or not state_file:
                        continue

                    try:
                        lfpHPC_r, hypno_r, _ = get_data(hpc_file, state_file)
                        lfpPFC_r, _, _ = get_data(pfc_file, state_file, type='pfc')
                        try:
                            phasic_int, tonic_int, lfp_raw_r = extract_pt_intervals(
                                lfpHPC_r, hypno_r, fs=fs)
                        except ValueError:
                            continue

                        if phasic_int is not None and len(phasic_int) > 0:
                            try:
                                ph_imfs, _, _ = extract_imfs_by_pt_intervals(
                                    lfp_raw_r, fs, phasic_int, cfg, return_imfs_freqs=True)
                                pfc_ph = extract_lfp_by_pt_intervals(lfpPFC_r, fs, phasic_int)
                                W_list = _extract_ppc_phase_diffs(
                                    ph_imfs, pfc_ph, fs, frequencies, n_phase_bins, n_cycles_wavelet)
                                phasic_W.extend(W_list)
                            except Exception as e:
                                print(f"  [WARN] Phasic: {e}")

                        if tonic_int is not None and len(tonic_int) > 0:
                            try:
                                to_imfs, _, _ = extract_imfs_by_pt_intervals(
                                    lfp_raw_r, fs, tonic_int, cfg, return_imfs_freqs=True)
                                pfc_to = extract_lfp_by_pt_intervals(lfpPFC_r, fs, tonic_int)
                                W_list = _extract_ppc_phase_diffs(
                                    to_imfs, pfc_to, fs, frequencies, n_phase_bins, n_cycles_wavelet)
                                tonic_W.extend(W_list)
                            except Exception as e:
                                print(f"  [WARN] Tonic: {e}")

                        print(f"  [OK] {condition_folder}/{pt_folder} "
                              f"(phasic: {len(phasic_W)}, tonic: {len(tonic_W)})")
                    except Exception as e:
                        print(f"  [ERROR] {condition_folder}/{pt_folder}: {e}")

        # Compute PPC(f, phi) per rat
        rat_result = {'frequencies': frequencies, 'n_phase_bins': n_phase_bins}
        for label, W_all in [('phasic', phasic_W), ('tonic', tonic_W)]:
            N = len(W_all)
            rat_result[f'{label}_n'] = N
            if N >= 2:
                sum_vec = np.zeros((len(frequencies), n_phase_bins), dtype=complex)
                for W_k in W_all:
                    sum_vec += np.exp(1j * W_k)
                ppc = (np.abs(sum_vec)**2 - N) / (N * (N - 1))
                rat_result[f'{label}_ppc'] = ppc
            else:
                rat_result[f'{label}_ppc'] = None

        print(f"\n  Rat {rat_id}: phasic={rat_result['phasic_n']}, tonic={rat_result['tonic_n']}")
        results[rat_id] = rat_result

    if save_path:
        with open(save_path, 'wb') as fp:
            pickle.dump(results, fp)
        print(f"\nSaved to {save_path}")

    return results

In [ ]:
# ---- Run phase-frequency PPC for the same rats as spectral_connectivity ----
selected_rats_pf = [1, 3, 6, 9]  # <-- match your selected_rats_sc

results_pf = compute_phase_freq_ppc_for_dataset(
    rat_ids=selected_rats_pf,
    save_path='ppc_phase_freq_results.pkl'
)

In [ ]:
def plot_phase_freq_ppc_per_rat(results_pf, vmin=-0.05, vmax=0.15):
    """Per-rat phase-frequency PPC heatmaps: phasic, tonic, difference."""
    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

    for rat_id, res in sorted(results_pf.items()):
        freqs = res['frequencies']
        ppc_ph = res['phasic_ppc']
        ppc_to = res['tonic_ppc']

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Rat {rat_id} — HPC-PFC Phase-Frequency PPC (wavelet)',
                     fontsize=14, fontweight='bold')

        if ppc_ph is not None:
            im = axes[0].pcolormesh(phase_centers, freqs, ppc_ph,
                                     cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
            axes[0].set_title(f'Phasic (n={res["phasic_n"]})')
            axes[0].set_ylabel('Frequency (Hz)')
            axes[0].set_xlabel('Theta phase (deg)')
            plt.colorbar(im, ax=axes[0], label='PPC')
        else:
            axes[0].set_title(f'Phasic (n={res["phasic_n"]}) — no data')

        if ppc_to is not None:
            im = axes[1].pcolormesh(phase_centers, freqs, ppc_to,
                                     cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
            axes[1].set_title(f'Tonic (n={res["tonic_n"]})')
            axes[1].set_ylabel('Frequency (Hz)')
            axes[1].set_xlabel('Theta phase (deg)')
            plt.colorbar(im, ax=axes[1], label='PPC')
        else:
            axes[1].set_title(f'Tonic (n={res["tonic_n"]}) — no data')

        if ppc_ph is not None and ppc_to is not None:
            diff = ppc_ph - ppc_to
            vd = np.max(np.abs(diff)) * 0.8
            im = axes[2].pcolormesh(phase_centers, freqs, diff,
                                     cmap='RdBu_r', shading='auto', vmin=-vd, vmax=vd)
            axes[2].set_title('Phasic - Tonic')
            axes[2].set_ylabel('Frequency (Hz)')
            axes[2].set_xlabel('Theta phase (deg)')
            plt.colorbar(im, ax=axes[2], label='$\\Delta$PPC')
        else:
            axes[2].set_title('Difference — insufficient data')

        plt.tight_layout()
        plt.show()


plot_phase_freq_ppc_per_rat(results_pf)

In [ ]:
def plot_grand_avg_phase_freq_ppc(results_pf, vmin=-0.05, vmax=0.15):
    """Grand-average phase-frequency PPC heatmaps across all rats."""
    phase_bins = np.linspace(-180, 180, 21)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2

    phasic_ppcs, tonic_ppcs = [], []
    freqs = None

    for rat_id, res in results_pf.items():
        freqs = res['frequencies']
        if res['phasic_ppc'] is not None:
            phasic_ppcs.append(res['phasic_ppc'])
        if res['tonic_ppc'] is not None:
            tonic_ppcs.append(res['tonic_ppc'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Grand Average Phase-Frequency PPC (n={len(results_pf)} rats)',
                 fontsize=14, fontweight='bold')

    if phasic_ppcs:
        ga_ph = np.mean(phasic_ppcs, axis=0)
        im = axes[0].pcolormesh(phase_centers, freqs, ga_ph,
                                 cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        axes[0].set_title(f'Phasic (n={len(phasic_ppcs)} rats)')
        axes[0].set_ylabel('Frequency (Hz)')
        axes[0].set_xlabel('Theta phase (deg)')
        plt.colorbar(im, ax=axes[0], label='PPC')
    else:
        axes[0].set_title('Phasic — no data')

    if tonic_ppcs:
        ga_to = np.mean(tonic_ppcs, axis=0)
        im = axes[1].pcolormesh(phase_centers, freqs, ga_to,
                                 cmap='RdBu_r', shading='auto', vmin=vmin, vmax=vmax)
        axes[1].set_title(f'Tonic (n={len(tonic_ppcs)} rats)')
        axes[1].set_ylabel('Frequency (Hz)')
        axes[1].set_xlabel('Theta phase (deg)')
        plt.colorbar(im, ax=axes[1], label='PPC')
    else:
        axes[1].set_title('Tonic — no data')

    if phasic_ppcs and tonic_ppcs:
        ga_diff = ga_ph - ga_to
        vd = np.max(np.abs(ga_diff)) * 0.8
        im = axes[2].pcolormesh(phase_centers, freqs, ga_diff,
                                 cmap='RdBu_r', shading='auto', vmin=-vd, vmax=vd)
        axes[2].set_title('Phasic - Tonic')
        axes[2].set_ylabel('Frequency (Hz)')
        axes[2].set_xlabel('Theta phase (deg)')
        plt.colorbar(im, ax=axes[2], label='$\\Delta$PPC')
    else:
        axes[2].set_title('Difference — insufficient data')

    plt.tight_layout()
    plt.show()


plot_grand_avg_phase_freq_ppc(results_pf)

# positive rats - wavelet version

In [ ]:
# ---- Choose which rats to run ----
# Options: 'all', or a list like [1, 2, 3]
selected_rats = [2, 4, 7, 8]  # <-- CHANGE THIS to select rats

results = compute_ppc_for_dataset(
    rat_ids=selected_rats,
    save_path='ppc_results.pkl'  # saves a pickle for later use
)

In [ ]:
plot_ppc_per_rat(results)

In [ ]:
plot_ppc_frequency_summary(results)

In [ ]:
plot_grand_average_ppc(results)